# 15 · Evaluación final

**Fase CRISP-DM:** 5 y 6. Evaluación y despliegue

**Objetivo:** medir **una sola vez** los modelos en la **prueba** (meses 11 y 12), que no se usó para ninguna decisión: ni features, ni hiperparámetros, ni umbrales. Es la estimación honesta de cómo rendiría el recomendador.

**Reglas fijadas antes de este notebook:**

- Modelos y umbrales **congelados**: LightGBM y logística del notebook 12/11, respaldo de popularidad con `N = 1` (notebook 13).
- Criterio de éxito: LightGBM debe superar a la mejor línea base por al menos **0.01 de NDCG@3** (`config/models.yaml`), con el intervalo de confianza por comensal arriba de ese valor.
- Los modelos **no se reentrenan** con train + validación: se evalúan tal como se eligieron.

> Este notebook solo narra el algoritmo. La lógica vive en `src/puzzle_recsys`.

## Configuración

In [1]:
import json

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from puzzle_recsys.catalog.library import load_library
from puzzle_recsys.config import load_paths, load_simulation, load_yaml
from puzzle_recsys.eda.multivariate import latent_diner_traits
from puzzle_recsys.evaluation.metrics import catalog_coverage, rank_metrics, summarize
from puzzle_recsys.evaluation.oracle import oracle_scores
from puzzle_recsys.evaluation.resampling import bootstrap_ci
from puzzle_recsys.evaluation.statistical_tests import paired_bootstrap, paired_permutation
from puzzle_recsys.explainability.ground_truth_check import truth_items
from puzzle_recsys.features.build import as_new_restaurant, without_history
from puzzle_recsys.features.calendar import MEAL_PERIODS
from puzzle_recsys.profiles.diner_profile import cuisine_prior
from puzzle_recsys.ranking.baselines import content_score, popularity_score, random_score
from puzzle_recsys.ranking.feature_sets import FOR_TABLE, FOR_YOU, model_frame
from puzzle_recsys.ranking.lgbm_ranker import predict
from puzzle_recsys.repository.parquet_repo import ParquetRepository, run_id_for
from puzzle_recsys.runtime import keep_awake
from puzzle_recsys.simulation.context import dish_period_bonus
from puzzle_recsys.simulation.distributions import make_rng

keep_awake()
paths, sim, mcfg, fcfg = load_paths(), load_simulation(), load_yaml("models"), load_yaml("features")
SEED, ecfg = sim.seed, mcfg["evaluation"]
MIN_GAIN, N_FALLBACK = ecfg["min_relevant_improvement"], fcfg["fallback"]["restaurant_min_orders"]
ti = pd.read_parquet(paths.data_model / "para_ti.parquet")
test = ti[ti["split"] == "test"].copy()
del ti
meta = json.loads((paths.models / "lgbm_meta.json").read_text(encoding="utf-8"))
lgbm = lgb.Booster(model_file=str(paths.models / "lgbm_para_ti.txt"))
logistica = joblib.load(paths.models / "logistica_para_ti.joblib")
cats = meta["para_ti"]["categories"]
cols = list(FOR_YOU.columns)
q_info = test.drop_duplicates("query_id").set_index("query_id")[
    ["diner_id", "course", "tramo_historia", "primera_vez_restaurante", "onboarding_completo"]]
pd.Series({"consultas de prueba": len(q_info), "candidatos": len(test), "comensales": q_info["diner_id"].nunique(),
           "meses": "11 y 12"}, name="valor").to_frame().T

,consultas de prueba,candidatos,comensales,meses
valor,81225,480335,5919,11 y 12


## 1. Métricas en la prueba, con intervalos de confianza

Intervalos bootstrap **por comensal** (2,000 remuestreos de personas, no de consultas sueltas: notebook 13).

In [2]:
def puntajes(df):
    return {"azar": random_score(df, seed=SEED).to_numpy(),
            "popularidad": popularity_score(df, min_orders=N_FALLBACK).to_numpy(),
            "contenido": content_score(df).to_numpy(),
            "logística": logistica.predict_proba(model_frame(df, FOR_YOU)[cols])[:, 1],
            "lgbm": predict(lgbm, df, FOR_YOU, cats)}

def metricas(df, scores, ks=(3, 5)):
    return {k: rank_metrics(df.assign(score=s), ks=ks, seed=SEED).join(q_info, on="query_id") for k, s in scores.items()}

s_test = puntajes(test)
pq = metricas(test, s_test)
filas = []
for k, v in pq.items():
    ci = bootstrap_ci(v["ndcg@3"], clusters=v["diner_id"], n_boot=ecfg["bootstrap_iterations"], seed=SEED)
    hit = bootstrap_ci(v["hit@3"], clusters=v["diner_id"], n_boot=ecfg["bootstrap_iterations"], seed=SEED)
    filas.append({"modelo": k, "NDCG@3": ci["media"], "IC NDCG@3": f"{ci['inferior']:.3f} a {ci['superior']:.3f}",
                  "HitRate@3": hit["media"], "IC HitRate@3": f"{hit['inferior']:.3f} a {hit['superior']:.3f}",
                  "MAP@3": v["map@3"].mean(), "NDCG@5": v["ndcg@5"].mean(),
                  "cobertura@3": catalog_coverage(test.assign(score=s_test[k]), k=3, seed=SEED)})
tabla = pd.DataFrame(filas).set_index("modelo")
tabla.round(4)

,NDCG@3,IC NDCG@3,HitRate@3,IC HitRate@3,MAP@3,NDCG@5,cobertura@3
modelo,,,,,,,
azar,0.4632,0.460 a 0.466,0.6064,0.603 a 0.610,0.4140,0.5455,0.9352
popularidad,0.6552,0.652 a 0.658,0.7924,0.789 a 0.795,0.6079,0.7048,0.6055
contenido,0.6518,0.649 a 0.655,0.7947,0.792 a 0.798,0.6024,0.7044,0.8361
logística,0.7032,0.700 a 0.706,0.8392,0.836 a 0.842,0.6562,0.7457,0.7294
lgbm,0.7353,0.733 a 0.738,0.8641,0.862 a 0.867,0.6907,0.7724,0.7871


In [3]:
comp = []
for a in ["popularidad", "logística"]:
    diff = pq["lgbm"]["ndcg@3"] - pq[a]["ndcg@3"]
    perm = paired_permutation(diff, pq[a]["diner_id"], n_iter=ecfg["permutation_iterations"], seed=SEED)
    boot = paired_bootstrap(diff, pq[a]["diner_id"], n_iter=ecfg["bootstrap_iterations"], seed=SEED)
    comp.append({"comparación": f"lgbm - {a}", "diferencia NDCG@3": boot["diferencia"],
                 "IC 95 %": f"{boot['inferior']:.4f} a {boot['superior']:.4f}", "p permutación": perm["p"],
                 f"¿supera el mínimo relevante ({MIN_GAIN})?": boot["inferior"] >= MIN_GAIN})
pd.DataFrame(comp).round(4)

,comparación,diferencia NDCG@3,IC 95 %,p permutación,¿supera el mínimo relevante (0.01)?
0,lgbm - popularidad,0.0800,0.0777 a 0.0825,0.0002,True
1,lgbm - logística,0.0321,0.0306 a 0.0336,0.0002,True


## 2. Techo del oráculo

¿Cuánto del camino posible recorre el modelo? El **oráculo** ordena cada consulta por la utilidad **verdadera** (gustos latentes conocidos). Ni el oráculo llega a 1, porque la elección simulada tiene una parte aleatoria: a veces la persona pide algo que no era su favorito.

Es un techo **aproximado**: incluye cocina, proteína, ingredientes, picante, atractivo, calidad y el bono por franja, pero no el término de precio (depende de un precio ideal por persona que no se guarda). El techo real es algo mayor. Se calcula en 5,000 consultas de prueba al azar.

In [4]:
repo = ParquetRepository(paths.synthetic / run_id_for(sim.run_name, sim.seed))
gt = repo.ground_truth
library = load_library(paths.external / "canonical_dishes.yaml")
cuisines = sorted(repo.read_table("restaurants")["cuisine_pred"].unique())
qs = make_rng(SEED, "oracle_sample").choice(q_info.index.to_numpy(), 5_000, replace=False)
sub = test[test["query_id"].isin(set(qs))]
menu = repo.read_table("menu_items").merge(gt.read_table("menu_items"), on="menu_item_id")
menu = menu[menu["menu_item_id"].isin(set(sub["menu_item_id"]))]
items = truth_items(menu, repo.read_table("canonical_dishes").set_index("canonical_id")["cuisine"],
                    repo.read_table("menu_item_attributes").query("kind == 'ingredient'"),
                    gt.read_table("canonical_appeal").set_index("canonical_id")["appeal"],
                    gt.read_table("menu_item_quality").set_index("menu_item_id")["quality"],
                    cuisines=cuisines, ingredient_names=list(library.ingredients))
tastes = latent_diner_traits(gt.read_table("diner_tastes"), gt.read_table("duplicate_accounts"),
                             pd.Index(sub["diner_id"].unique()), cuisines)
canon_true = sub["menu_item_id"].map(menu.set_index("menu_item_id")["canonical_true"])
bonus = dish_period_bonus(canon_true.to_numpy(), library, sim.choice["context"])
periodo = sub["franja"].astype(str).map({p: k for k, p in enumerate(MEAL_PERIODS)}).to_numpy()
bonus_fila = bonus[np.arange(len(sub)), periodo]
oraculo = oracle_scores(sub, tastes, items, sim.choice, period_bonus=bonus_fila)

techo = {"azar": s_test["azar"], "popularidad": s_test["popularidad"], "logística": s_test["logística"], "lgbm": s_test["lgbm"]}
mask = test["query_id"].isin(set(qs)).to_numpy()
res_o = {k: summarize(rank_metrics(sub.assign(score=v[mask]), ks=(3,), seed=SEED))["ndcg@3"].iloc[0] for k, v in techo.items()}
res_o["oráculo"] = summarize(rank_metrics(sub.assign(score=oraculo), ks=(3,), seed=SEED))["ndcg@3"].iloc[0]
o = pd.Series(res_o, name="NDCG@3").to_frame()
o["% del camino (azar → oráculo)"] = 100 * (o["NDCG@3"] - res_o["azar"]) / (res_o["oráculo"] - res_o["azar"])
o.round(3)

,NDCG@3,% del camino (azar → oráculo)
azar,0.461,0.000
popularidad,0.660,68.392
logística,0.709,85.100
lgbm,0.742,96.326
oráculo,0.752,100.000


## 3. Resultados por segmento

### 3.1 Tipo de plato, historia y primera visita al restaurante

In [5]:
todos = pd.concat([v.assign(modelo=k) for k, v in pq.items()], ignore_index=True)
orden = list(pq)
for seg in ["course", "tramo_historia", "primera_vez_restaurante", "onboarding_completo"]:
    t = summarize(todos, by=["modelo", seg]).pivot(index=seg, columns="modelo", values="ndcg@3")[orden]
    t.insert(0, "consultas", summarize(pq["lgbm"], by=seg).set_index(seg)["consultas"])
    display(t.round(3))

modelo,consultas,azar,popularidad,contenido,logística,lgbm
course,,,,,,
DESSERT,7819,0.690,0.814,0.786,0.834,0.853
DRINK,27428,0.424,0.652,0.589,0.684,0.724
MAIN,32332,0.307,0.536,0.585,0.616,0.651
SIDE,2277,0.905,0.936,0.927,0.941,0.949
STARTER,11369,0.757,0.836,0.846,0.861,0.878


modelo,consultas,azar,popularidad,contenido,logística,lgbm
tramo_historia,,,,,,
0,4,0.283,0.908,0.658,0.908,0.908
1-5,141,0.517,0.682,0.598,0.707,0.750
6-20,11458,0.467,0.667,0.624,0.702,0.737
21+,69622,0.463,0.653,0.656,0.703,0.735


modelo,consultas,azar,popularidad,contenido,logística,lgbm
primera_vez_restaurante,,,,,,
False,59827,0.459,0.654,0.665,0.707,0.738
True,21398,0.475,0.659,0.616,0.693,0.727


modelo,consultas,azar,popularidad,contenido,logística,lgbm
onboarding_completo,,,,,,
False,31863,0.462,0.653,0.648,0.701,0.733
True,49362,0.464,0.657,0.654,0.704,0.737


### 3.2 Persona nueva y restaurante nuevo (contrafactuales, notebook 13)

In [6]:
restaurants = repo.read_table("restaurants")
share = restaurants["cuisine_pred"].value_counts(normalize=True).reindex(cuisines, fill_value=0)
prior = cuisine_prior(repo.read_table("diner_profiles"), share, declared_weight=fcfg["profile"]["declared_weight"])
escenarios = {"real (con historia)": test, "persona nueva": without_history(test, prior),
              "restaurante nuevo": as_new_restaurant(test)}
filas = {}
for nombre, df in escenarios.items():
    sc = puntajes(df)
    filas[nombre] = {k: summarize(rank_metrics(df.assign(score=v), ks=(3,), seed=SEED))["ndcg@3"].iloc[0]
                     for k, v in sc.items() if k != "contenido"}
pd.DataFrame(filas).T[["azar", "popularidad", "logística", "lgbm"]].round(4)

,azar,popularidad,logística,lgbm
real (con historia),0.4632,0.6552,0.7032,0.7353
persona nueva,0.4632,0.6552,0.6609,0.6919
restaurante nuevo,0.4632,0.6402,0.6717,0.7098


### 3.3 "Para la mesa"

In [7]:
mesa = pd.read_parquet(paths.data_model / "para_la_mesa.parquet")
mte = mesa[mesa["split"] == "test"]
lm = lgb.Booster(model_file=str(paths.models / "lgbm_para_la_mesa.txt"))
log_mesa = joblib.load(paths.models / "logistica_para_la_mesa.joblib")
hosts = repo.read_table("guests").query("is_host")[["session_id", "diner_id"]]
mq = mte.drop_duplicates("query_id")[["query_id", "session_id"]].merge(hosts, on="session_id").set_index("query_id")
sm = {"azar": random_score(mte, seed=SEED), "popularidad de compartir": mte["tasa_compartido_canonico"],
      "logística": log_mesa.predict_proba(model_frame(mte, FOR_TABLE)[list(FOR_TABLE.columns)])[:, 1],
      "lgbm": predict(lm, mte, FOR_TABLE, meta["para_la_mesa"]["categories"])}
fm = []
for k, s in sm.items():
    r = rank_metrics(mte.assign(score=np.asarray(s)), ks=(1, 3), seed=SEED).join(mq, on="query_id")
    ci = bootstrap_ci(r["hit@1"], clusters=r["diner_id"], n_boot=ecfg["bootstrap_iterations"], seed=SEED)
    fm.append({"modelo": k, "mesas": len(r), "Hit@1": ci["media"], "IC Hit@1": f"{ci['inferior']:.3f} a {ci['superior']:.3f}",
               "NDCG@3": r["ndcg@3"].mean()})
pd.DataFrame(fm).set_index("modelo").round(3)

,mesas,Hit@1,IC Hit@1,NDCG@3
modelo,,,,
azar,5776,0.619,0.606 a 0.631,0.823
popularidad de compartir,5776,0.742,0.730 a 0.754,0.882
logística,5776,0.815,0.805 a 0.826,0.917
lgbm,5776,0.831,0.821 a 0.841,0.922


## 4. Plan de despliegue en Puzzle

### 4.1 Arquitectura

| Paso | Cuándo | Qué hace |
|---|---|---|
| Homologación | al dar de alta un platillo | sugiere el canónico (KNN, notebook 09); el restaurante confirma. Con confianza < 0.7, se pide confirmación antes de usarlo |
| Perfil a la fecha | cada noche | recalcula el perfil de cada comensal y la popularidad con lo ocurrido hasta ayer |
| Ranking | al abrir el menú | filtro duro de alergias → features de unos 50 platillos → LightGBM → lista "para ti" por tipo de plato y "para la mesa" |
| Respaldo | siempre | si el modelo falla o el restaurante no tiene historia: popularidad del canónico en la plataforma (`N = 1`) |

Los cambios de base de datos que esto requiere están en `docs/cambios-base-de-datos.md` (identidad persistente del comensal, canónico y alérgenos del platillo, historial de precios y disponibilidad, bandera de compartido y calificaciones). **El modelo se entrena con datos simulados: no se despliega tal cual.**

### 4.2 Camino a producción

1. **Recolectar** al menos 3 meses de datos reales con las tablas nuevas, mostrando popularidad (lo que ya rinde 0.66 de NDCG@3 en la simulación).
2. **Recalibrar primero lo supuesto, no estimado:** contexto por franja (S16), compartir según la mesa (S17) y las bandas de validación (S14). Repetir los notebooks 04 a 08 con datos reales.
3. **Reentrenar** con el mismo pipeline (09 a 13) sobre los datos reales.
4. **Prueba A/B:** popularidad (control) contra LightGBM (tratamiento), aleatorizando **por comensal**. Según la potencia del notebook 13, unos **1,000 comensales activos por brazo** bastan para detectar una mejora de 0.01 de NDCG@3. Métrica principal: platillo pedido entre los 3 primeros mostrados (HitRate@3 en línea).
5. **Monitorear:** HitRate@3 semanal por tipo de plato, cobertura del catálogo (sesgo de popularidad) y proporción de recomendaciones con canónico `SUGGESTED` (sin confirmar).

## Conclusiones

### 1. Veredicto: LightGBM cumple el criterio de éxito en la prueba

| Modelo | NDCG@3 (IC 95 %) | HitRate@3 | Cobertura@3 |
|---|---|---|---|
| Azar | 0.463 (0.460 a 0.466) | 0.606 | 0.94 |
| Popularidad | 0.655 (0.652 a 0.658) | 0.792 | 0.61 |
| Contenido | 0.652 (0.649 a 0.655) | 0.795 | 0.84 |
| Regresión logística | 0.703 (0.700 a 0.706) | 0.839 | 0.73 |
| **LightGBM** | **0.735 (0.733 a 0.738)** | **0.864** | **0.79** |

- **Sobre la popularidad:** +0.080 de NDCG@3 (IC 0.078 a 0.083). **Sobre la logística:** +0.032 (IC 0.031 a 0.034). Las dos superan con holgura la mejora mínima relevante (0.01), con p < 0.001 en la permutación por comensal.
- **Lo pedido queda en el top 3 en el 86 % de las consultas**, contra 79 % de la popularidad.
- **Los números de la prueba repiten los de validación** (0.735 contra 0.737): no hubo sobreajuste a la validación.
- **Menos sesgo de popularidad:** LightGBM muestra el 79 % del catálogo en algún top 3, contra 61 % de la popularidad.

### 2. Cerca del techo

Contra el **oráculo aproximado** (ordenar por la utilidad verdadera, sin el término de precio), LightGBM recorre el **96 %** del camino entre el azar y el techo (0.742 contra 0.752 en 5,000 consultas). La logística recorre el 85 % y la popularidad el 68 %. **Advertencia:** el techo real es algo mayor que el aproximado, porque le falta el precio, y LightGBM aprende parte de ese efecto a través de la popularidad. El 96 % es un **límite superior** del avance real, no una medición exacta. Aun así, queda poco margen para modelos más complejos **con estas features**. Lo que más podría mejorar está en el notebook 14: afinidad por cocina con los platillos pedidos y afinidad por ingrediente.

### 3. Por segmento

- **Tipo de plato:** gana en todos. Más donde hay más opciones: bebida (0.684 → 0.724 sobre la logística) y plato fuerte (0.616 → 0.651).
- **Primera vez en el restaurante** (26 % de las consultas; el segmento principal de la propuesta): **0.727** contra 0.659 de la popularidad. El conocimiento se transfiere entre restaurantes.
- **Persona nueva** (contrafactual): **0.692**, sigue arriba de la popularidad (0.655). La historia vale unos 0.04.
- **Restaurante nuevo** (contrafactual): **0.710** contra 0.640 de la popularidad con respaldo global.
- **Onboarding completo:** ayuda poco (0.737 contra 0.733).

### 4. "Para la mesa"

LightGBM acierta el platillo compartido en el primer lugar en el **83 %** de las mesas (IC 0.821 a 0.841), contra 74 % de la popularidad de compartir y 62 % del azar. La ventaja sobre la logística (0.831 contra 0.815) es chica. Con 2 candidatos por mesa (mediana), es un problema casi resuelto por cualquier modelo que use la composición de la mesa.

### 5. Lo que estos números **no** dicen

- **Es rendimiento sobre datos simulados.** Valida el método (sin fuga, con pruebas honestas y potencia suficiente), no cuánto acertará en Puzzle. Eso solo lo dice la **prueba A/B** del plan de despliegue.
- Los efectos de contexto (S16) y de compartir (S17) son **supuestos**: lo que el modelo aprendió de ellos refleja la simulación y se recalibra primero.

### Cierre del ciclo CRISP-DM

| Fase | Notebooks |
|---|---|
| 1. Entendimiento del negocio | 00 |
| 2. Entendimiento de los datos | 01 a 08 |
| 3. Preparación | 09 y 10 |
| 4. Modelado | 11 y 12 |
| 5. Evaluación | 13 a 15 |
| 6. Despliegue | 15 (plan) y `docs/cambios-base-de-datos.md` |
